# UNet-Lite — Human Segmentation (COCO 2017)

Notebook điều phối **mỏng**: toàn bộ logic nằm trong package `src/` (model, dataset, losses, metrics, engine, visualize). Notebook chỉ gọi API.

**Pipeline:** UNet-Lite (khối MobileNetV2) · Mixed Precision (fp16) · Gradient Accumulation (batch hiệu dụng 256) · CosineAnnealingLR · Early Stopping.

> Lưu ý: chạy notebook từ **thư mục gốc dự án** để import được `cv_nets`.

## 1. Thiết lập & import

In [ ]:
import os, sys
# Đảm bảo thư mục gốc dự án nằm trong sys.path (để import src và cv_nets)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src import (
    Config, build_model, build_dataloaders, build_loss, fit, load_checkpoint,
)
from src.model import count_parameters
from src import visualize as viz
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Cấu hình

Mọi siêu tham số gom trong `Config` (xem `src/config.py`). Chỉnh tại đây nếu cần.

In [ ]:
cfg = Config()
# Ví dụ tinh chỉnh:
# cfg.epochs = 30
# cfg.loss_name = 'focal_dice'   # focal_dice | bce_dice | ohem_dice
# cfg.num_workers = 8
print('Thiết bị :', cfg.device)
print('Ảnh      :', cfg.image_size, '| Batch:', cfg.batch_size,
      '| Accum:', cfg.accumulation_steps, '-> batch hiệu dụng', cfg.batch_size*cfg.accumulation_steps)
print('Loss     :', cfg.loss_name, '| Epochs:', cfg.epochs, '| LR:', cfg.lr)

## 3. Dữ liệu (COCO person)

In [ ]:
train_loader, val_loader = build_dataloaders(cfg)
print(f'Train: {len(train_loader.dataset)} ảnh | Val: {len(val_loader.dataset)} ảnh chứa người')

In [ ]:
viz.show_batch(train_loader, num_samples=8)

In [ ]:
viz.show_batch(val_loader, num_samples=8)

## 4. Mô hình UNet-Lite

In [ ]:
model = build_model(cfg)
print(f'Tham số học được: {count_parameters(model):,}')

## 5. Huấn luyện

`fit` lo trọn: AMP, grad-accum, cosine LR, early stopping, lưu best/last checkpoint, ghi history.

In [ ]:
criterion = build_loss(cfg)
# Train tiếp từ checkpoint: fit(..., resume=cfg.last_ckpt)
history = fit(model, train_loader, val_loader, criterion, cfg)

## 6. Đường cong huấn luyện

In [ ]:
viz.plot_history(history)   # hoặc: viz.plot_history(cfg.history_file)

## 7. Đánh giá & trực quan hoá

Nạp lại mô hình tốt nhất rồi so sánh Input | Ground Truth | Prediction.

In [ ]:
model = build_model(cfg).to(cfg.device)
load_checkpoint(cfg.best_ckpt, model, map_location=cfg.device)
viz.show_predictions(model, val_loader, cfg, num_samples=8)

In [ ]:
viz.show_predictions(model, val_loader, cfg, num_samples=8, colored=True)

In [ ]:
viz.show_predictions(model, train_loader, cfg, num_samples=8, colored=True)